# File Handling in Python
### A Python Buddy Guide

---

File handling lets your Python programs read data from files and write results back to disk — essential for working with real-world data.

**This notebook covers:**
1. Opening and closing files
2. Reading files — `read()`, `readline()`, `readlines()`
3. Writing files — `write()`, `writelines()`
4. The `with` statement (context manager)
5. File modes — r, w, a, x, r+, w+
6. Working with file paths
7. Checking if a file exists
8. Reading structured data (CSV)
9. Applied example — OPPE-style question walkthrough

---
## 1. Opening and Closing Files

The built-in `open()` function opens a file and returns a **file object**.

```python
file = open(filename, mode)
```

Always **close** the file when done — leaving it open wastes resources and can corrupt data.

In [ ]:
# Create a sample file to work with throughout this notebook
import os

sample_content = """Alice 85
Bob 72
Carol 91
David 60
Eve 88
"""

with open("students.txt", "w") as f:
    f.write(sample_content)

print("Sample file created: students.txt")

In [ ]:
# Manual open and close — NOT recommended
f = open("students.txt", "r")
content = f.read()
f.close()   # ← easy to forget!

print(content)

---
## 2. Reading Files

Three methods to read from an open file:

| Method | Returns | Use when |
|--------|---------|----------|
| `read()` | Entire file as one string | Small files, need full content |
| `readline()` | One line at a time | Large files, line-by-line processing |
| `readlines()` | List of all lines | Need random access to lines |

In [ ]:
# read() — entire file as a single string
with open("students.txt", "r") as f:
    content = f.read()

print(repr(content))   # repr() shows \n characters

In [ ]:
# readline() — one line per call
with open("students.txt", "r") as f:
    line1 = f.readline()   # 'Alice 85\n'
    line2 = f.readline()   # 'Bob 72\n'
    line3 = f.readline()   # 'Carol 91\n'

print(repr(line1))
print(repr(line2))
print(repr(line3))

In [ ]:
# readlines() — all lines as a list
with open("students.txt", "r") as f:
    lines = f.readlines()

print(lines)
print(f"\nNumber of lines: {len(lines)}")
print(f"Third line: {repr(lines[2])}")

In [ ]:
# Iterating over a file directly — most memory-efficient
# Works for very large files — reads one line at a time
with open("students.txt", "r") as f:
    for line in f:
        print(line.strip())   # strip() removes the trailing \n

---
## 3. Writing Files

Two methods to write:

| Method | Behaviour |
|--------|----------|
| `write(string)` | Writes a single string — no automatic newline |
| `writelines(list)` | Writes each item in the list — no automatic newlines |

In [ ]:
# write() — write a string
with open("output.txt", "w") as f:
    f.write("Hello, World!\n")
    f.write("Second line\n")
    f.write("Third line\n")

# Verify
with open("output.txt", "r") as f:
    print(f.read())

In [ ]:
# writelines() — write a list of strings
lines = ["apple\n", "banana\n", "cherry\n"]

with open("fruits.txt", "w") as f:
    f.writelines(lines)   # note: no \n added automatically!

with open("fruits.txt") as f:
    print(f.read())

In [ ]:
# Appending to an existing file — mode 'a'
with open("fruits.txt", "a") as f:
    f.write("durian\n")
    f.write("elderberry\n")

with open("fruits.txt") as f:
    print(f.read())

---
## 4. The `with` Statement (Context Manager)

The `with` statement automatically closes the file when the block ends — even if an error occurs.

```python
with open(filename, mode) as f:
    # work with f here
# file is automatically closed here
```

> **Always use `with`** instead of manual `open()`/`close()`. It's safer and cleaner.

In [ ]:
# Comparing manual vs with

# ❌ Manual — error-prone
f = open("students.txt")
data = f.read()
f.close()   # what if an exception happened before this? file stays open!

# ✅ with — always closes, even on error
with open("students.txt") as f:
    data = f.read()
# f is guaranteed closed here

print("File closed:", f.closed)

In [ ]:
# Opening multiple files at once with with
with open("students.txt") as fin, open("results.txt", "w") as fout:
    for line in fin:
        name, score = line.strip().split()
        grade = "Pass" if int(score) >= 75 else "Fail"
        fout.write(f"{name}: {grade}\n")

with open("results.txt") as f:
    print(f.read())

---
## 5. File Modes

| Mode | Name | Creates file? | Truncates? | Read | Write |
|------|------|:---:|:---:|:---:|:---:|
| `'r'` | Read (default) | ❌ | ❌ | ✅ | ❌ |
| `'w'` | Write | ✅ | ✅ | ❌ | ✅ |
| `'a'` | Append | ✅ | ❌ | ❌ | ✅ |
| `'x'` | Exclusive create | ✅ (fails if exists) | ❌ | ❌ | ✅ |
| `'r+'` | Read + write | ❌ | ❌ | ✅ | ✅ |
| `'w+'` | Write + read | ✅ | ✅ | ✅ | ✅ |

Add `'b'` for binary mode: `'rb'`, `'wb'` etc.

In [ ]:
# 'r' — fails if file doesn't exist
try:
    with open("nonexistent.txt", "r") as f:
        pass
except FileNotFoundError as e:
    print(f"Error: {e}")

In [ ]:
# 'w' — overwrites existing content!
with open("demo.txt", "w") as f:
    f.write("Original content\n")

with open("demo.txt", "w") as f:
    f.write("Replaced!\n")   # original content is GONE

with open("demo.txt") as f:
    print(f.read())

In [ ]:
# 'x' — creates file, raises FileExistsError if already exists
import os

if os.path.exists("new_file.txt"):
    os.remove("new_file.txt")

with open("new_file.txt", "x") as f:
    f.write("Created fresh\n")

try:
    with open("new_file.txt", "x") as f:   # already exists!
        pass
except FileExistsError as e:
    print(f"Error: {e}")

---
## 6. Working with File Paths

The `os.path` module and the modern `pathlib` library help you work with file paths in a platform-independent way.

In [ ]:
import os

# Current working directory
print("CWD:", os.getcwd())

# Join paths safely (works on Windows and Linux)
path = os.path.join("folder", "subfolder", "file.txt")
print("Joined path:", path)

# Split a path
full_path = "/home/user/documents/report.txt"
print("Directory:", os.path.dirname(full_path))
print("Filename:",  os.path.basename(full_path))
print("Split:",     os.path.split(full_path))
print("Extension:", os.path.splitext(full_path))

In [ ]:
# Modern approach — pathlib (Python 3.4+)
from pathlib import Path

p = Path("students.txt")
print("Exists:",    p.exists())
print("Stem:",      p.stem)       # filename without extension
print("Suffix:",    p.suffix)     # extension
print("Parent:",    p.parent)     # directory

# Read entire file with pathlib
content = p.read_text()
print("\nContent:")
print(content)

---
## 7. Checking if a File Exists

Always check before opening — especially in `'r'` mode — to avoid `FileNotFoundError`.

In [ ]:
import os

# Using os.path.exists()
if os.path.exists("students.txt"):
    with open("students.txt") as f:
        print(f.read())
else:
    print("File not found")

In [ ]:
# Using try-except — more Pythonic (EAFP style)
# EAFP = Easier to Ask Forgiveness than Permission
try:
    with open("students.txt") as f:
        print(f.readline().strip())
except FileNotFoundError:
    print("File not found")

In [ ]:
# List all files in a directory
import os

files = os.listdir(".")
txt_files = [f for f in files if f.endswith(".txt")]
print("Text files in current directory:")
for f in txt_files:
    print(" ", f)

---
## 8. Reading Structured Data (CSV)

CSV (Comma-Separated Values) is one of the most common file formats. Python has a built-in `csv` module for reading and writing CSV files properly — handling quoted fields, commas inside values, etc.

In [ ]:
# Create a sample CSV file
csv_content = """name,score,grade
Alice,85,B
Bob,72,C
Carol,91,A
David,60,D
Eve,88,B
"""

with open("students.csv", "w") as f:
    f.write(csv_content)

print("CSV file created")

In [ ]:
import csv

# Reading CSV with csv.reader
with open("students.csv") as f:
    reader = csv.reader(f)
    header = next(reader)   # skip header row
    print("Columns:", header)
    for row in reader:
        name, score, grade = row
        print(f"{name}: {score} ({grade})")

In [ ]:
# csv.DictReader — each row as a dictionary
with open("students.csv") as f:
    reader = csv.DictReader(f)
    for row in reader:
        print(f"{row['name']} scored {row['score']}")

In [ ]:
# Writing CSV with csv.writer
data = [
    ["name", "score", "result"],
    ["Alice", 85, "Pass"],
    ["Bob", 72, "Pass"],
    ["David", 60, "Fail"],
]

with open("results.csv", "w", newline="") as f:
    writer = csv.writer(f)
    writer.writerows(data)

with open("results.csv") as f:
    print(f.read())

---
## 9. Applied Example — OPPE-Style Question

### Question: Student Report Generator

Given a file `marks.txt` where each line contains a student name and their marks (space-separated), write a program that:
1. Reads the file
2. Computes the average marks
3. Classifies each student as `Pass` (≥ 50) or `Fail` (< 50)
4. Writes a report to `report.txt` with one line per student: `name: marks — Pass/Fail`
5. Prints the class average and the topper's name

In [ ]:
# Setup — create input file
marks_data = """Alice 85
Bob 42
Carol 91
David 55
Eve 38
Frank 76
"""
with open("marks.txt", "w") as f:
    f.write(marks_data)

In [ ]:
# ── Solution ─────────────────────────────────────────────────

# Step 1: Read all student data
students = []
with open("marks.txt") as f:
    for line in f:
        line = line.strip()
        if line:                          # skip blank lines
            name, marks = line.split()
            students.append((name, int(marks)))

print("Read:", students)

In [ ]:
# Step 2: Compute average
average = sum(m for _, m in students) / len(students)
print(f"Class average: {average:.2f}")

# Step 3 + 4: Classify and write report
with open("report.txt", "w") as f:
    f.write(f"Class Average: {average:.2f}\n")
    f.write("-" * 30 + "\n")
    for name, marks in students:
        result = "Pass" if marks >= 50 else "Fail"
        f.write(f"{name}: {marks} — {result}\n")

# Step 5: Print topper
topper = max(students, key=lambda s: s[1])
print(f"Topper: {topper[0]} with {topper[1]} marks")

# Show the report
print("\n--- report.txt ---")
with open("report.txt") as f:
    print(f.read())

In [ ]:
# Clean up all demo files
import os
for fname in ["students.txt", "students.csv", "results.csv", "results.txt",
              "output.txt", "fruits.txt", "demo.txt", "new_file.txt",
              "marks.txt", "report.txt"]:
    if os.path.exists(fname):
        os.remove(fname)
print("All demo files cleaned up.")

---
## Summary

| Concept | Key point |
|---------|----------|
| `open(file, mode)` | Opens a file — always use `with` |
| `read()` | Entire file as one string |
| `readline()` | One line at a time |
| `readlines()` | All lines as a list |
| Iterating `for line in f` | Most memory-efficient for large files |
| `write(s)` | Writes string — no automatic newline |
| `writelines(lst)` | Writes list of strings — no automatic newlines |
| Mode `'r'` | Read only (default) |
| Mode `'w'` | Write — **overwrites** existing file |
| Mode `'a'` | Append — adds to end of file |
| Mode `'x'` | Create — fails if file already exists |
| `with` statement | Auto-closes file — always use it |
| `os.path.exists()` | Check before opening to avoid errors |
| `csv.reader` | Read CSV row by row as lists |
| `csv.DictReader` | Read CSV row by row as dicts |

> **Golden rule:** Always use `with open(...) as f` — it guarantees the file is closed even if something goes wrong.